## TRYING TO CATCH THE BEST RISK-MANAGEMENT

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "MultiagentSystem" / "multiagent_predictions_module.py").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError(
        "Не найден корень репозитория (папка MultiagentSystem). "
        "Запускайте ноутбук из Risk_management_calculation или из корня проекта."
    )


_REPO = _repo_root()
sys.path.insert(0, str(_REPO))

_here = Path.cwd().resolve()
_candidates = [
    _here / "predictions_results.csv",
    _REPO / "Risk_management_calculation" / "predictions_results_tech_twitter_v2.csv",
    _REPO / "MultiagentSystem" / "predictions_results.csv",
]
_csv_path = next((p for p in _candidates if p.exists()), None)
if _csv_path is None:
    raise FileNotFoundError(
        "Не найден predictions_results.csv. Ожидался один из путей:\n"
        + "\n".join(str(p) for p in _candidates)
    )

df = pd.read_csv(_csv_path, parse_dates=["forecast_start_date"])
df = df.sort_values("forecast_start_date").reset_index(drop=True)

_horizon_cols = ("horizon_close_price", "horizon_min_low", "horizon_max_high")
if not all(c in df.columns for c in _horizon_cols):
    from dotenv import load_dotenv

    load_dotenv(_REPO / "dev.env")
    from MultiagentSystem.multiagent_predictions_module import add_y_true

    with open(_REPO / "configs" / "multiagent_config.json", encoding="utf-8") as f:
        _cfg = json.load(f)
    df = add_y_true(df, int(_cfg["horizon"]))

df["price_change"] = df["horizon_close_price"] - df["start_date_price"]
df["volatility_in_low_in_percent"] = abs(df["start_date_price"] - df["horizon_min_low"]) / df["start_date_price"] * 100
df["volatility_in_high_in_percent"] = abs(df["horizon_max_high"] - df["start_date_price"]) / df["start_date_price"] * 100

TEST_FRACTION = 0.20
n_test = max(int(round(len(df) * TEST_FRACTION)), 1)
TRAIN = df.iloc[:-n_test].copy()
TEST = df.iloc[-n_test:].copy()

print("CSV:", _csv_path)
print(
    "TRAIN:",
    TRAIN["forecast_start_date"].min().date(),
    "->",
    TRAIN["forecast_start_date"].max().date(),
    len(TRAIN),
)
print(
    "TEST :",
    TEST["forecast_start_date"].min().date(),
    "->",
    TEST["forecast_start_date"].max().date(),
    len(TEST),
)


CSV: D:\Actives_prices_prediction\Actives_price_prediction\Risk_management_calculation\predictions_results.csv
TRAIN: 2026-01-12 -> 2026-04-09 88
TEST : 2026-04-10 -> 2026-05-01 22


In [2]:
import os
print(os.listdir("."))


['.ipynb_checkpoints', 'predictions_results.csv', 'predictions_results_tech_twitter_v2.csv', 'RM_calc.ipynb']


In [3]:
TEST[["y_predict", "y_true", "price_change", "start_date_price", "volatility_in_low_in_percent", "volatility_in_high_in_percent"]].head(len(TEST))

,y_predict,y_true,price_change,start_date_price,volatility_in_low_in_percent,volatility_in_high_in_percent
88,LONG,LONG,1260.5,71788.0,1.016883,2.815373
89,LONG,SHORT,-2212.7,72955.4,3.359587,0.253580
90,LONG,LONG,1366.7,73048.9,3.391564,2.528169
91,LONG,SHORT,3401.7,70742.7,4.333451,7.502258
92,LONG,LONG,397.8,74415.1,1.205535,1.368271
93,LONG,LONG,1005.2,74144.2,1.149382,1.879715
94,LONG,LONG,2247.1,74812.8,0.376941,4.725261
95,LONG,SHORT,542.9,75150.1,0.380971,3.013569
96,LONG,SHORT,-3257.0,77060.0,4.289904,1.058915
97,LONG,LONG,139.6,75693.3,2.618462,1.148979


In [4]:
import numpy as np
import pandas as pd

# ---------- ПАРАМЕТРЫ СТРАТЕГИИ ----------
FEE_PCT          = 0.001    # комиссия биржи за одну сторону сделки
SLIPPAGE_PCT     = 0.0001   # проскальзывание market-ордера при выходе

STOP_LOSS_PCT_LONG    = 2.0      # SL для LONG в %
TAKE_PROFIT_PCT_LONG  = 5.0     # TP для LONG в %

STOP_LOSS_PCT_SHORT   = 2.0     # SL для SHORT в %
TAKE_PROFIT_PCT_SHORT = 5.0     # TP для SHORT в %

# COMBINED:  LONG TP/SL = 4.75/1.5  |  SHORT TP/SL = 4.5/2.25

POS_SIZE_PCT     = 0.2     # доля депозита в одну сделку

# ---------- НАЧАЛЬНОЕ СОСТОЯНИЕ ----------
budget       = 1000.0
equity_curve = [budget]
sl_hits, tp_hits, close_hits, skipped_double = 0, 0, 0, 0
total_fees, total_slip = 0.0, 0.0

# ---------- ОСНОВНОЙ ЦИКЛ ПО TEST ----------
trade_num = 0
for _, row in TRAIN.iterrows():
    y = row["y_predict"]
    if pd.isna(y) or y not in ("LONG", "SHORT"):
        continue

    conf = row["y_predict_confidence"]
    if pd.isna(conf):
        continue

    start  = row["start_date_price"]
    close  = row["horizon_close_price"]
    vol_hi = row["volatility_in_high_in_percent"]
    vol_lo = row["volatility_in_low_in_percent"]

    if any(pd.isna(x) for x in (start, close, vol_hi, vol_lo)):
        continue

    notional = budget * POS_SIZE_PCT
    if notional <= 0:
        break

    entry_fee = notional * FEE_PCT

    # --- выбор TP/SL по стороне сделки ---
    if y == "LONG":
        sl_pct, tp_pct = STOP_LOSS_PCT_LONG,  TAKE_PROFIT_PCT_LONG
        sl_hit = vol_lo >= sl_pct
        tp_hit = vol_hi >= tp_pct
    else:  # SHORT
        sl_pct, tp_pct = STOP_LOSS_PCT_SHORT, TAKE_PROFIT_PCT_SHORT
        sl_hit = vol_hi >= sl_pct
        tp_hit = vol_lo >= tp_pct

    if sl_hit and tp_hit:
        skipped_double += 1
        date_str = row.get('forecast_start_date', '?')
        print(f"[SKIP double-hit] {date_str} {y} start={start:.2f} close={close:.2f} hi={vol_hi:.2f}% lo={vol_lo:.2f}%")
        continue
    elif sl_hit:
        pct = -sl_pct
    elif tp_hit:
        pct = tp_pct
    else:
        pct = (close / start - 1) * 100 if y == "LONG" else (1 - close / start) * 100

    exit_price_ratio = (1 + pct / 100) if y == "LONG" else (1 - pct / 100)

    if   sl_hit: sl_hits += 1;    exit_reason = "SL"
    elif tp_hit: tp_hits += 1;    exit_reason = "TP"
    else:        close_hits += 1; exit_reason = "CLOSE"

    pnl           = notional * (pct / 100)
    exit_notional = notional * exit_price_ratio
    exit_fee      = exit_notional * FEE_PCT
    slip          = notional * SLIPPAGE_PCT

    net_pnl       = pnl - entry_fee - exit_fee - slip
    budget_before = budget
    budget       += net_pnl
    total_fees   += entry_fee + exit_fee
    total_slip   += slip
    equity_curve.append(budget)

    trade_num += 1
    date_str  = row.get('forecast_start_date', '?')
    conf_val  = row['y_predict_confidence']
    print(
        f"#{trade_num:>3} {date_str} {y:<5} conf={conf_val:+.1f} | "
        f"start={start:>9.2f} close={close:>9.2f} hi=+{vol_hi:>5.2f}% lo=-{vol_lo:>5.2f}% | "
        f"exit={exit_reason:<5} TP={tp_pct:.2f}% SL={sl_pct:.2f}% | "
        f"notional=${notional:>8.2f} pnl=${pnl:+8.2f} fee=${entry_fee+exit_fee:>5.2f} slip=${slip:>5.2f} "
        f"budget: ${budget_before:>8.2f} -> ${budget:>8.2f}"
    )

print(f"\nИтого сделок: {trade_num} | TP={tp_hits} SL={sl_hits} CLOSE={close_hits} | пропущено double-hit: {skipped_double}")
print(f"Финальный депозит: ${budget:.2f} (начальный $1000)")
print(f"Суммарные комиссии: ${total_fees:.2f} | суммарный slippage: ${total_slip:.2f}")


#  1 2026-01-12 00:00:00 LONG  conf=+1.0 | start= 91004.10 close= 95412.80 hi=+ 6.15% lo=- 0.05% | exit=TP    TP=5.00% SL=2.00% | notional=$  200.00 pnl=$  +10.00 fee=$ 0.41 slip=$ 0.02 budget: $ 1000.00 -> $ 1009.57
[SKIP double-hit] 2026-01-13 00:00:00 LONG start=91295.10 close=96950.80 hi=7.29% lo=3.55%
#  2 2026-01-14 00:00:00 LONG  conf=+1.0 | start= 95412.90 close= 95599.70 hi=+ 1.87% lo=- 0.30% | exit=CLOSE TP=5.00% SL=2.00% | notional=$  201.91 pnl=$   +0.40 fee=$ 0.40 slip=$ 0.02 budget: $ 1009.57 -> $ 1009.54
#  3 2026-01-15 00:00:00 LONG  conf=+1.0 | start= 96950.80 close= 95550.60 hi=+ 1.13% lo=- 2.74% | exit=SL    TP=5.00% SL=2.00% | notional=$  201.91 pnl=$   -4.04 fee=$ 0.40 slip=$ 0.02 budget: $ 1009.54 -> $ 1005.08
#  4 2026-01-16 00:00:00 LONG  conf=+1.0 | start= 95599.70 close= 95144.00 hi=+ 0.05% lo=- 0.60% | exit=CLOSE TP=5.00% SL=2.00% | notional=$  201.02 pnl=$   -0.96 fee=$ 0.40 slip=$ 0.02 budget: $ 1005.08 -> $ 1003.70
#  5 2026-01-17 00:00:00 LONG  conf=+1.0 

In [5]:
# double_hits = 0
# for _, row in TEST.iterrows():
#     y = row["y_predict"]
#     if y not in ("LONG", "SHORT"):
#         continue
#     vol_hi = row["volatility_in_high_in_percent"]
#     vol_lo = row["volatility_in_low_in_percent"]
#     if pd.isna(vol_hi) or pd.isna(vol_lo):
#         continue
#     if vol_hi >= TAKE_PROFIT_PCT and vol_lo >= STOP_LOSS_PCT:
#         double_hits += 1

# print(f"Дней с двойным попаданием: {double_hits}")

In [6]:
import matplotlib.pyplot as plt

plt.plot(equity_curve)
plt.title("Equity curve")
plt.xlabel("Trade №")
plt.ylabel("Equity, $")
plt.grid()
plt.show()

C:\Users\flays\AppData\Local\Temp\ipykernel_21528\2643644817.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
first = df.iloc[0]
last = df.iloc[-1]

# Buy-and-hold: close → close на датах первого и последнего прогноза (без смешения open и close).
entry_price = first["btc_bybit_close_price"]
exit_price = last["btc_bybit_close_price"]

pnl_abs = exit_price - entry_price
pnl_pct = pnl_abs / entry_price * 100
days = (last["forecast_start_date"] - first["forecast_start_date"]).days

print(f"Вход : {first['forecast_start_date'].date()}  @ {entry_price:,.2f}")
print(f"Выход: {last['forecast_start_date'].date()}   @ {exit_price:,.2f}")
print(f"Дней удержания: {days} (строк: {len(df)})")
print(f"PnL: {pnl_abs:+,.2f} USD ({pnl_pct:+.2f} %)")

capital = 1000
print(f"$ {capital} -> $ {capital * (1 + pnl_pct / 100):,.2f}")

Вход : 2026-01-12  @ 91,295.10
Выход: 2026-05-01   @ 78,044.60
Дней удержания: 109 (строк: 110)
PnL: -13,250.50 USD (-14.51 %)
$ 1000 -> $ 854.86


In [8]:
import numpy as np
import pandas as pd
from itertools import product


def simulate_tp_sl(
    df: pd.DataFrame,
    tp: float = None,
    sl: float = None,
    *,
    tp_long: float = None,
    sl_long: float = None,
    tp_short: float = None,
    sl_short: float = None,
    min_abs_confidence: float = 0.0,
    fee_pct: float = 0.001,
    slippage_pct: float = 0.0001,
    pos_size_pct: float = 1.0,
    double_hit_policy: str = "skip",
    initial_budget: float = 1000.0,
    compounding: bool = True,
) -> dict:
    """
    Симулирует стратегию по предсказаниям с TP / SL (в %), отдельными для LONG и SHORT.

    Симметричный режим:  передать tp/sl  -> используется и для LONG, и для SHORT.
    Асимметричный режим: передать tp_long/sl_long/tp_short/sl_short.
    Любой *_long/*_short, оставшийся None, наследуется от tp/sl.
    """
    if tp_long  is None: tp_long  = tp
    if sl_long  is None: sl_long  = sl
    if tp_short is None: tp_short = tp
    if sl_short is None: sl_short = sl
    if any(x is None for x in (tp_long, sl_long, tp_short, sl_short)):
        raise ValueError("Need either tp+sl or all of tp_long/sl_long/tp_short/sl_short")

    budget = initial_budget
    equity_curve = [budget]
    pnl_list, exit_reasons, sides = [], [], []
    tp_hits = sl_hits = close_hits = skipped_double = 0
    long_tp = long_sl = long_close = 0
    short_tp = short_sl = short_close = 0
    total_fees = total_slip = 0.0

    for _, row in df.iterrows():
        side = row["y_predict"]
        if side not in ("LONG", "SHORT"):
            continue
        conf = row["y_predict_confidence"]
        if pd.isna(conf) or abs(conf) < min_abs_confidence:
            continue

        start = row["start_date_price"]
        close = row["horizon_close_price"]
        vol_hi = row["volatility_in_high_in_percent"]
        vol_lo = row["volatility_in_low_in_percent"]
        if any(pd.isna(x) for x in (start, close, vol_hi, vol_lo)):
            continue

        cur_tp, cur_sl = (tp_long, sl_long) if side == "LONG" else (tp_short, sl_short)

        if side == "LONG":
            hit_sl = vol_lo >= cur_sl
            hit_tp = vol_hi >= cur_tp
        else:
            hit_sl = vol_hi >= cur_sl
            hit_tp = vol_lo >= cur_tp

        if hit_sl and hit_tp:
            if double_hit_policy == "skip":
                skipped_double += 1
                continue
            elif double_hit_policy == "sl":
                pct, reason = -cur_sl, "SL"
            elif double_hit_policy == "tp":
                pct, reason = +cur_tp, "TP"
            else:
                raise ValueError(f"unknown double_hit_policy: {double_hit_policy!r}")
        elif hit_sl:
            pct, reason = -cur_sl, "SL"
        elif hit_tp:
            pct, reason = +cur_tp, "TP"
        else:
            pct = (close / start - 1) * 100 if side == "LONG" else (1 - close / start) * 100
            reason = "CLOSE"

        if reason == "TP":   tp_hits += 1;    long_tp    += (side == "LONG"); short_tp    += (side == "SHORT")
        elif reason == "SL": sl_hits += 1;    long_sl    += (side == "LONG"); short_sl    += (side == "SHORT")
        else:                close_hits += 1; long_close += (side == "LONG"); short_close += (side == "SHORT")

        notional = (budget if compounding else initial_budget) * pos_size_pct
        if notional <= 0:
            break

        exit_ratio = (1 + pct / 100) if side == "LONG" else (1 - pct / 100)
        pnl_gross = notional * (pct / 100)
        entry_fee = notional * fee_pct
        exit_fee  = notional * exit_ratio * fee_pct
        slip      = notional * slippage_pct
        pnl_net   = pnl_gross - entry_fee - exit_fee - slip

        budget += pnl_net
        equity_curve.append(budget)
        total_fees += entry_fee + exit_fee
        total_slip += slip

        pnl_list.append(pnl_net / notional * 100)
        exit_reasons.append(reason)
        sides.append(side)

    n = len(pnl_list)
    base = dict(
        tp_long=tp_long, sl_long=sl_long, tp_short=tp_short, sl_short=sl_short,
        skipped_double=skipped_double,
        final_budget=budget, total_fees=total_fees, total_slippage=total_slip,
        equity_curve=equity_curve,
    )
    if n == 0:
        return {**base,
            "trades": 0, "total_return": 0.0, "avg_return": 0.0,
            "win_rate": 0.0, "sharpe": 0.0, "max_drawdown": 0.0,
            "tp_hits": 0, "sl_hits": 0, "close_exits": 0,
            "long_trades": 0, "long_tp": 0, "long_sl": 0, "long_close": 0, "long_win_rate": 0.0,
            "short_trades": 0, "short_tp": 0, "short_sl": 0, "short_close": 0, "short_win_rate": 0.0,
            "tp_trivial": False,
        }

    pnl = np.array(pnl_list)
    sides_arr = np.array(sides)
    equity_arr = np.array(equity_curve)
    drawdown = (equity_arr / np.maximum.accumulate(equity_arr) - 1).min() * 100
    total_return = (budget / initial_budget - 1) * 100
    sharpe = pnl.mean() / pnl.std() if pnl.std() > 0 else 0.0  # per-trade

    is_long  = sides_arr == "LONG"
    is_short = sides_arr == "SHORT"
    long_n, short_n = int(is_long.sum()), int(is_short.sum())
    long_wr  = float((pnl[is_long]  > 0).mean() * 100) if long_n  else 0.0
    short_wr = float((pnl[is_short] > 0).mean() * 100) if short_n else 0.0

    return {**base,
        "trades": n,
        "total_return": total_return,
        "avg_return": pnl.mean(),
        "win_rate": (pnl > 0).mean() * 100,
        "sharpe": sharpe,
        "max_drawdown": drawdown,
        "tp_hits": tp_hits, "sl_hits": sl_hits, "close_exits": close_hits,
        "long_trades": long_n,  "long_tp": long_tp,   "long_sl": long_sl,   "long_close": long_close,   "long_win_rate": long_wr,
        "short_trades": short_n,"short_tp": short_tp, "short_sl": short_sl, "short_close": short_close, "short_win_rate": short_wr,
        "tp_trivial": (tp_hits == n and n > 0),
    }


def grid_search_one_side(
    df: pd.DataFrame,
    side: str,
    tp_range=(0.5, 5.0, 0.25),
    sl_range=(0.5, 5.0, 0.25),
    *,
    optimize: str = "total_return",
    **sim_kwargs,
) -> pd.DataFrame:
    """
    Грид TP / SL по одной стороне (LONG или SHORT).
    Берёт срез df только с нужной стороной, гридит TP/SL для неё одной,
    оптимизирует по `optimize` (по умолчанию total_return — максимизация дохода).
    """
    assert side in ("LONG", "SHORT"), side
    sub = df[df["y_predict"] == side]

    tp_values = np.round(np.arange(tp_range[0], tp_range[1] + 1e-9, tp_range[2]), 4)
    sl_values = np.round(np.arange(sl_range[0], sl_range[1] + 1e-9, sl_range[2]), 4)

    rows = []
    for tp_v, sl_v in product(tp_values, sl_values):
        r = simulate_tp_sl(
            sub,
            tp_long=tp_v,
            sl_long=sl_v,
            tp_short=tp_v,
            sl_short=sl_v,
            **sim_kwargs,
        )
        r["tp"], r["sl"] = tp_v, sl_v
        rows.append(r)

    grid = pd.DataFrame(rows).drop(columns=["equity_curve"], errors="ignore")
    grid = grid.sort_values(
        by=[optimize, "sl_hits", "max_drawdown", "sl"],
        ascending=[False, True, False, True],
    )
    grid = grid.drop_duplicates(subset=["total_return", "win_rate", "tp_hits", "sl_hits"], keep="first")
    return grid.reset_index(drop=True)


In [9]:
TRAIN.columns

Index(['forecast_start_date', 'y_predict', 'y_predict_confidence', 'summary',
       'reasoning', 'risks', 'analysing_tech_indicators__prediction',
       'analysing_tech_indicators__confidence', 'twitter_analysis__prediction',
       'twitter_analysis__confidence', 'start_date_price',
       'btc_bybit_close_price', 'btc_bybit_high_price', 'btc_bybit_low_price',
       'y_true', 'horizon_close_price', 'horizon_max_high', 'horizon_min_low',
       'price_change', 'volatility_in_low_in_percent',
       'volatility_in_high_in_percent'],
      dtype='object')

In [11]:
SIM_KWARGS = dict(
    min_abs_confidence=1.0,
    fee_pct=0.001,
    slippage_pct=0.0001,
    pos_size_pct=0.2,
    double_hit_policy="skip",
    compounding=True,
)

# 1) ПОДБОР TP/SL ОТДЕЛЬНО ДЛЯ КАЖДОЙ СТОРОНЫ — максимизируем total_return
TP_SL_RANGE = (0.5, 5.0, 0.25)

subset = TRAIN[
    TRAIN['y_predict'].notna()
    & (TRAIN['y_predict'] != '')
].copy()


long_grid  = grid_search_one_side(subset, "LONG",  TP_SL_RANGE, TP_SL_RANGE,
                                  optimize="total_return", **SIM_KWARGS)
short_grid = grid_search_one_side(subset, "SHORT", TP_SL_RANGE, TP_SL_RANGE,
                                  optimize="total_return", **SIM_KWARGS)

best_long  = long_grid.iloc[0]
best_short = short_grid.iloc[0]

print("=== TRAIN | LONG-ONLY (TP-сигналы модели) ===")
print(long_grid[["tp", "sl", "trades", "total_return", "win_rate",
                 "tp_hits", "sl_hits", "close_exits", "max_drawdown", "tp_trivial"]].head(10).to_string(index=False))
print(f"\nЛучшая LONG-пара: TP={best_long.tp}%  SL={best_long.sl}%  "
      f"trades={int(best_long.trades)}  return={best_long.total_return:+.2f}%  "
      f"win_rate={best_long.win_rate:.1f}%  DD={best_long.max_drawdown:.2f}%")
if bool(best_long.tp_trivial):
    print("  [WARNING] LONG TP-trivial: SL ни разу не сработал — возможен overfit")

print("\n=== TRAIN | SHORT-ONLY (SHORT-сигналы модели) ===")
print(short_grid[["tp", "sl", "trades", "total_return", "win_rate",
                  "tp_hits", "sl_hits", "close_exits", "max_drawdown", "tp_trivial"]].head(10).to_string(index=False))
print(f"\nЛучшая SHORT-пара: TP={best_short.tp}%  SL={best_short.sl}%  "
      f"trades={int(best_short.trades)}  return={best_short.total_return:+.2f}%  "
      f"win_rate={best_short.win_rate:.1f}%  DD={best_short.max_drawdown:.2f}%")
if bool(best_short.tp_trivial):
    print("  [WARNING] SHORT TP-trivial: SL ни разу не сработал — возможен overfit")

# 2) КОМБИНИРУЕМ В ЕДИНУЮ СТРАТЕГИЮ И ВАЛИДИРУЕМ НА TEST
print("\n" + "=" * 70)
print(f"COMBINED:  LONG TP/SL = {best_long.tp}/{best_long.sl}  |  "
      f"SHORT TP/SL = {best_short.tp}/{best_short.sl}")
print("=" * 70)

train_combined = simulate_tp_sl(
    TRAIN,
    tp_long=best_long.tp,   sl_long=best_long.sl,
    tp_short=best_short.tp, sl_short=best_short.sl,
    **SIM_KWARGS,
)
test_combined = simulate_tp_sl(
    TEST,
    tp_long=best_long.tp,   sl_long=best_long.sl,
    tp_short=best_short.tp, sl_short=best_short.sl,
    **SIM_KWARGS,
)

def _print_summary(name, r):
    print(f"\n--- {name} ---")
    print(f"  trades       = {r['trades']}  (LONG {r['long_trades']} / SHORT {r['short_trades']})")
    print(f"  total_return = {r['total_return']:+.2f}%   final ${r['final_budget']:.2f}  (start $1000)")
    print(f"  win_rate     = {r['win_rate']:.1f}%  "
          f"(LONG {r['long_win_rate']:.1f}% | SHORT {r['short_win_rate']:.1f}%)")
    print(f"  exits        = TP {r['tp_hits']} / SL {r['sl_hits']} / CLOSE {r['close_exits']}  "
          f"skipped_double={r['skipped_double']}")
    print(f"  LONG  exits  = TP {r['long_tp']} / SL {r['long_sl']} / CLOSE {r['long_close']}")
    print(f"  SHORT exits  = TP {r['short_tp']} / SL {r['short_sl']} / CLOSE {r['short_close']}")
    print(f"  sharpe (per-trade) = {r['sharpe']:.3f}   max DD = {r['max_drawdown']:.2f}%")
    print(f"  fees=${r['total_fees']:.2f}  slip=${r['total_slippage']:.2f}")

_print_summary("TRAIN (combined per-side)", train_combined)
_print_summary("TEST  (combined per-side)", test_combined)


=== TRAIN | LONG-ONLY (TP-сигналы модели) ===
  tp   sl  trades  total_return  win_rate  tp_hits  sl_hits  close_exits  max_drawdown  tp_trivial
4.00 1.00      24      0.675472 33.333333        3       12            9     -1.730102       False
4.25 1.00      25      0.581479 32.000000        3       13            9     -1.730102       False
3.75 1.00      24      0.525818 33.333333        3       12            9     -1.730102       False
3.00 1.00      23      0.393061 34.782609        5       11            7     -1.730102       False
4.00 1.75      26      0.378698 38.461538        5       12            9     -2.758758       False
3.50 1.00      24      0.376312 33.333333        3       12            9     -1.730102       False
5.00 0.75      25      0.349029 28.000000        2       15            8     -1.565717       False
4.00 0.75      23      0.336852 30.434783        2       13            8     -1.565717       False
5.00 1.00      25      0.331378 32.000000        2       13    

In [ ]:
print("Распределение total_return по уровню TP (медиана по SL) — LONG:")
print(long_grid.groupby("tp")[["total_return", "win_rate"]].median().round(3).to_string())

print("\nРаспределение total_return по уровню TP (медиана по SL) — SHORT:")
print(short_grid.groupby("tp")[["total_return", "win_rate"]].median().round(3).to_string())


## BUILDING LOGISTIC REGRESSION FOR SEARCHING PATTERNS FOR REJECTING VOICES

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


def _repo_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "MultiagentSystem" / "multiagent_predictions_module.py").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise RuntimeError("Не найден корень репозитория (MultiagentSystem).")


_REPO = _repo_root()
_here = Path.cwd().resolve()
_candidates = [
    _here / "predictions_results.csv",
    _REPO / "Risk_management_calculation" / "predictions_results.csv",
    _REPO / "MultiagentSystem" / "predictions_results.csv",
]
_csv_path = next((p for p in _candidates if p.exists()), None)
if _csv_path is None:
    raise FileNotFoundError("Не найден predictions_results.csv.")

df = pd.read_csv(_csv_path)
df = df.dropna(subset=["y_predict", "y_true"]).sort_values("forecast_start_date").reset_index(drop=True)

CONF_MAP = {"low": 1, "medium": 2, "high": 3}

def signed(pred_col, conf_col):
    sign = df[pred_col].map({True: 1, False: -1, "True": 1, "False": -1})  # NaN → NaN
    mag  = df[conf_col].map(CONF_MAP)                        # NaN → NaN
    return (sign * mag).fillna(0).astype(float)              # abstain → 0

X = pd.DataFrame({
    "final_score":    df["y_predict_confidence"].astype(float),
    "abs_final":      df["y_predict_confidence"].abs().astype(float),
    "tech_signed":    signed("analysing_tech_indicators__prediction", "analysing_tech_indicators__confidence"),
    "twitter_signed": signed("twitter_analysis__prediction", "twitter_analysis__confidence"),
    "is_long_pred":   (df["y_predict"] == "LONG").astype(int),
})
y = (df["y_predict"] == df["y_true"]).astype(int)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score,
)

# ============================================================
# 1) Модель
# ============================================================
def make_pipeline():
    # SimpleImputer не нужен — все NaN уже превращены в 0 на этапе фичей.
    # C=0.1 даёт сильную L2-регуляризацию: критично при ~97 строках и 5 фичах.
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000,
            solver="liblinear",
        )),
    ])

# ============================================================
# 2) Walk-forward CV: собираем OOS-предсказания по всему датасету
# ============================================================
tscv = TimeSeriesSplit(n_splits=4)
oos_proba = np.full(len(X), np.nan)        # вероятность "прогноз верный"
oos_pred  = np.full(len(X), np.nan)        # бинарное решение по threshold=0.5 (для отчёта)
fold_metrics = []

for fold, (tr, te) in enumerate(tscv.split(X), start=1):
    pipe = make_pipeline()
    pipe.fit(X.iloc[tr], y.iloc[tr])

    proba_te = pipe.predict_proba(X.iloc[te])[:, 1]
    pred_te  = (proba_te >= 0.5).astype(int)

    oos_proba[te] = proba_te
    oos_pred[te]  = pred_te

    auc = roc_auc_score(y.iloc[te], proba_te) if len(set(y.iloc[te])) > 1 else float("nan")
    fold_metrics.append({
        "fold": fold,
        "n_train": len(tr),
        "n_test":  len(te),
        "acc":       accuracy_score(y.iloc[te], pred_te),
        "precision": precision_score(y.iloc[te], pred_te, zero_division=0),
        "recall":    recall_score(y.iloc[te], pred_te, zero_division=0),
        "auc":       auc,
    })

print("Per-fold OOS metrics (threshold=0.5):")
print(pd.DataFrame(fold_metrics).to_string(index=False))

# ============================================================
# 3) Поиск оптимального порога P_correct по precision на всех OOS-точках
# ============================================================
mask = ~np.isnan(oos_proba)        # первые len(tr) точек первого фолда не имеют OOS прогноза
y_oos    = y[mask].values
prob_oos = oos_proba[mask]

print(f"\nOOS pool: {mask.sum()} predictions ({y_oos.mean():.1%} positive baseline)")

rows = []
for thr in np.arange(0.40, 0.81, 0.05):
    take = prob_oos >= thr
    if take.sum() == 0:
        continue
    rows.append({
        "threshold":      round(thr, 2),
        "n_taken":        int(take.sum()),
        "coverage":       take.mean(),
        "precision":      precision_score(y_oos[take], np.ones(take.sum()), zero_division=0),
        "n_correct":      int(y_oos[take].sum()),
        "n_wrong":        int((1 - y_oos[take]).sum()),
    })
threshold_table = pd.DataFrame(rows)
print("\nThreshold sweep on OOS pool:")
print(threshold_table.to_string(index=False))

# ============================================================
# 4) Сравнение с baseline (брать ВСЁ что произвёл мультиагент)
# ============================================================
baseline_acc = y_oos.mean()  # доля правильных, если фильтра нет
print(f"\nBaseline (no filter) accuracy on OOS pool: {baseline_acc:.1%}")
print("→ фильтр имеет смысл только если есть строки threshold_table.precision > baseline_acc + 3pp")

# ============================================================
# 5) Финальная модель на всём датасете (для использования на проде)
# ============================================================
final_model = make_pipeline().fit(X, y)
print("\nFitted coefficients (standardized scale):")
coefs = pd.Series(final_model.named_steps["clf"].coef_[0], index=X.columns).sort_values()
print(coefs.to_string())
print(f"intercept: {final_model.named_steps['clf'].intercept_[0]:.3f}")

# ============================================================
# 6) Confusion matrix на OOS при threshold=0.5 (для интерпретации)
# ============================================================
pred_oos = (prob_oos >= 0.5).astype(int)
cm = confusion_matrix(y_oos, pred_oos, labels=[0, 1])
print(f"\nConfusion matrix (OOS, threshold=0.5):")
print(f"                       pred=skip   pred=take")
print(f"actual=wrong (y=0)     {cm[0,0]:>5}      {cm[0,1]:>5}")
print(f"actual=correct (y=1)   {cm[1,0]:>5}      {cm[1,1]:>5}")
